# Entrenamiento SAC en CartPole-v1 con Ray RLlib

Este notebook convierte el script `run_debug.py` en una experiencia interactiva para entrenar un agente SAC (Soft Actor-Critic) en el entorno CartPole-v1 usando Ray RLlib y PyTorch.

## Objetivos:
- Configurar y entrenar un agente SAC
- Monitorear el progreso del entrenamiento
- Evaluar el rendimiento del agente
- Visualizar los resultados

## 0. Instalación de dependencias

**⚠️ Ejecuta esta celda solo si necesitas instalar las dependencias.**

Si ya tienes un entorno configurado, puedes saltar a la siguiente sección.

In [ ]:
# Descomenta y ejecuta las siguientes líneas si necesitas instalar las dependencias

# Opción 1: Instalar desde requirements_debug.txt (recomendado)
# !pip install -r requirements_debug.txt

# Opción 2: Instalar librerías principales manualmente
# !pip install ray[default]==2.47.1 torch==2.7.1 gymnasium==1.0.0 numpy matplotlib pandas

# Opción 3: Para instalación mínima (solo lo esencial)
# !pip install ray torch gymnasium numpy matplotlib

print("📦 Para instalar las dependencias, descomenta una de las opciones de arriba y ejecuta la celda")
print("📋 Las dependencias específicas están en: requirements_debug.txt")
print()
print("🔍 Dependencias principales necesarias:")
print("  • ray>=2.40.0 (framework de entrenamiento distribuido)")
print("  • torch>=2.0.0 (framework de deep learning)")
print("  • gymnasium>=1.0.0 (entornos de RL)")
print("  • numpy, matplotlib, pandas (análisis y visualización)")
print()
print("⚡ Si estás en Google Colab, puedes ejecutar directamente:")
print("   !pip install ray[default] torch gymnasium matplotlib")

In [ ]:
# Verificar que las librerías principales estén instaladas correctamente
try:
    import ray
    import torch
    import gymnasium
    import numpy
    import matplotlib
    import pandas
    
    print("✅ Verificación de instalación:")
    print(f"  • Ray: {ray.__version__}")
    print(f"  • PyTorch: {torch.__version__}")
    print(f"  • Gymnasium: {gymnasium.__version__}")
    print(f"  • NumPy: {numpy.__version__}")
    print(f"  • Matplotlib: {matplotlib.__version__}")
    print(f"  • Pandas: {pandas.__version__}")
    print()
    
    # Verificar disponibilidad de GPU
    if torch.cuda.is_available():
        print(f"🚀 GPU disponible: {torch.cuda.get_device_name(0)}")
        print(f"   CUDA version: {torch.version.cuda}")
    else:
        print("💻 Usando CPU (GPU no disponible)")
    
    print()
    print("🎉 ¡Todas las librerías están correctamente instaladas!")
    
except ImportError as e:
    print(f"❌ Error de importación: {e}")
    print("💡 Ejecuta la celda anterior para instalar las dependencias faltantes")

## 1. Importar librerías necesarias

Importamos todas las librerías requeridas para el entrenamiento del agente SAC.

In [ ]:
import ray
from ray import tune
from ray.rllib.algorithms.sac import SACConfig
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import os
from pathlib import Path

print("✅ Librerías importadas exitosamente")

## 2. Definir parámetros de configuración

Configuramos los hiperparámetros para el entrenamiento del agente SAC.

In [ ]:
# Parámetros del entorno
ENV_NAME = "CartPole-v1"
STORAGE_PATH = "/tmp/ray_results"
EXPERIMENT_NAME = "sac_cartpole_notebook"

# Parámetros de entrenamiento
TRAINING_CONFIG = {
    "timesteps_total": 50000,
    "episode_reward_target": 400,  # CartPole se considera resuelto con reward >= 400
    "train_batch_size": 256,
    "learning_rate": 3e-4,
    "replay_buffer_capacity": 50000,
    "num_env_runners": 2,
    "checkpoint_frequency": 10,
}

# Parámetros del replay buffer
REPLAY_BUFFER_CONFIG = {
    "type": "MultiAgentPrioritizedReplayBuffer",
    "capacity": TRAINING_CONFIG["replay_buffer_capacity"],
    "prioritized_replay_alpha": 0.6,
    "prioritized_replay_beta": 0.4,
    "prioritized_replay_eps": 1e-6,
}

# Parámetros de SAC
SAC_CONFIG = {
    "tau": 0.005,
    "target_network_update_freq": 1,
    "initial_alpha": 1.0,
    "target_entropy": "auto",
}

print("📋 Configuración definida:")
print(f"  Entorno: {ENV_NAME}")
print(f"  Timesteps objetivo: {TRAINING_CONFIG['timesteps_total']:,}")
print(f"  Reward objetivo: {TRAINING_CONFIG['episode_reward_target']}")
print(f"  Batch size: {TRAINING_CONFIG['train_batch_size']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")

## 3. Inicializar entorno y agente SAC

Configuramos Ray, creamos el entorno y configuramos el agente SAC con los parámetros definidos.

In [ ]:
# Inicializar Ray
try:
    ray.shutdown()  # Cerrar sesión previa si existe
except:
    pass

ray.init(ignore_reinit_error=True, log_to_driver=False)
print("🚀 Ray inicializado correctamente")
print(f"📊 Dashboard de Ray: {ray.get_dashboard_url()}")

In [ ]:
# Configurar el agente SAC
sac_config = (
    SACConfig()
    .environment(ENV_NAME)
    .framework("torch")
    .api_stack(
        enable_rl_module_and_learner=False,
        enable_env_runner_and_connector_v2=False,
    )
    .env_runners(
        num_env_runners=TRAINING_CONFIG["num_env_runners"],
        num_envs_per_env_runner=1,
    )
    .training(
        train_batch_size=TRAINING_CONFIG["train_batch_size"],
        replay_buffer_config=REPLAY_BUFFER_CONFIG,
        lr=TRAINING_CONFIG["learning_rate"],
        tau=SAC_CONFIG["tau"],
        target_network_update_freq=SAC_CONFIG["target_network_update_freq"],
        initial_alpha=SAC_CONFIG["initial_alpha"],
        target_entropy=SAC_CONFIG["target_entropy"],
    )
    .debugging(seed=42)
)

print("🤖 Configuración del agente SAC creada")
print("📝 Parámetros principales:")
print(f"  - Framework: torch")
print(f"  - Env runners: {TRAINING_CONFIG['num_env_runners']}")
print(f"  - Batch size: {TRAINING_CONFIG['train_batch_size']}")
print(f"  - Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Replay buffer: {REPLAY_BUFFER_CONFIG['capacity']:,} samples")

## 4. Entrenar el agente

Ejecutamos el entrenamiento del agente SAC con monitoreo del progreso en tiempo real.

In [ ]:
# Definir las condiciones de parada
stop_conditions = {
    "timesteps_total": TRAINING_CONFIG["timesteps_total"],
    "env_runners/episode_return_mean": TRAINING_CONFIG["episode_reward_target"],
}

print("🎯 Iniciando entrenamiento con las siguientes condiciones de parada:")
print(f"  - Timesteps máximos: {TRAINING_CONFIG['timesteps_total']:,}")
print(f"  - Reward objetivo: {TRAINING_CONFIG['episode_reward_target']}")
print("  - El entrenamiento se detendrá cuando se alcance cualquiera de las dos condiciones")
print()

# Ejecutar entrenamiento
training_start_time = time.time()

results = tune.run(
    "SAC",
    config=sac_config.to_dict(),
    stop=stop_conditions,
    checkpoint_config=tune.CheckpointConfig(
        checkpoint_score_attribute="env_runners/episode_return_mean",
        checkpoint_score_order="max",
        num_to_keep=3,
        checkpoint_frequency=TRAINING_CONFIG["checkpoint_frequency"],
        checkpoint_at_end=True
    ),
    storage_path=STORAGE_PATH,
    name=EXPERIMENT_NAME,
    verbose=1,
    resume="AUTO+ERRORED",
)

training_end_time = time.time()
training_duration = training_end_time - training_start_time

print()
print("🎉 Entrenamiento completado!")
print(f"⏱️  Duración total: {training_duration:.2f} segundos ({training_duration/60:.1f} minutos)")
print(f"📁 Resultados guardados en: {STORAGE_PATH}/{EXPERIMENT_NAME}")

## 5. Evaluar el agente entrenado

Cargamos el mejor checkpoint y evaluamos el rendimiento del agente entrenado.

In [ ]:
# Obtener métricas del entrenamiento
best_trial = results.get_best_trial("env_runners/episode_return_mean", "max")
best_checkpoint_path = results.get_best_checkpoint(best_trial, "env_runners/episode_return_mean", "max")

print("📈 Resultados del entrenamiento:")
print(f"  - Mejor trial: {best_trial}")
print(f"  - Mejor checkpoint: {best_checkpoint_path}")
print()

# Obtener métricas finales
final_metrics = best_trial.last_result
print("🎯 Métricas finales:")
print(f"  - Timesteps totales: {final_metrics.get('timesteps_total', 'N/A'):,}")
print(f"  - Reward promedio: {final_metrics.get('env_runners/episode_return_mean', 'N/A'):.2f}")
print(f"  - Reward máximo: {final_metrics.get('env_runners/episode_return_max', 'N/A'):.2f}")
print(f"  - Longitud promedio de episodio: {final_metrics.get('env_runners/episode_len_mean', 'N/A'):.2f}")
print(f"  - Iteraciones de entrenamiento: {final_metrics.get('training_iteration', 'N/A')}")

# Verificar si el problema se resolvió
target_reward = TRAINING_CONFIG["episode_reward_target"]
achieved_reward = final_metrics.get('env_runners/episode_return_mean', 0)

if achieved_reward >= target_reward:
    print(f"🎉 ¡Problema resuelto! Reward promedio ({achieved_reward:.2f}) >= objetivo ({target_reward})")
else:
    print(f"⚠️  Problema no resuelto. Reward promedio ({achieved_reward:.2f}) < objetivo ({target_reward})")

In [ ]:
# Cargar y probar el agente entrenado
from ray.rllib.algorithms.algorithm import Algorithm

# Restaurar el algoritmo desde el mejor checkpoint
agent = Algorithm.from_checkpoint(best_checkpoint_path.path)

# Crear entorno para evaluación
eval_env = gym.make(ENV_NAME)

print("🧪 Evaluando agente entrenado...")
print("Ejecutando 5 episodios de prueba:")

eval_rewards = []
eval_lengths = []

for episode in range(5):
    obs, _ = eval_env.reset()
    episode_reward = 0
    episode_length = 0
    done = False
    
    while not done:
        # Obtener acción del agente
        action = agent.compute_single_action(obs, explore=False)
        
        # Ejecutar acción en el entorno
        obs, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        
        episode_reward += reward
        episode_length += 1
    
    eval_rewards.append(episode_reward)
    eval_lengths.append(episode_length)
    
    print(f"  Episodio {episode + 1}: Reward = {episode_reward:.1f}, Longitud = {episode_length}")

eval_env.close()

# Calcular estadísticas de evaluación
mean_reward = np.mean(eval_rewards)
std_reward = np.std(eval_rewards)
mean_length = np.mean(eval_lengths)

print()
print("📊 Estadísticas de evaluación:")
print(f"  - Reward promedio: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  - Longitud promedio: {mean_length:.1f}")
print(f"  - Reward mínimo: {min(eval_rewards):.1f}")
print(f"  - Reward máximo: {max(eval_rewards):.1f}")

## 6. Visualizar resultados

Creamos gráficos para visualizar el progreso del entrenamiento y el rendimiento del agente.

In [ ]:
# Obtener datos del progreso del entrenamiento
trial_dataframe = best_trial.get_trial_dataframe()

# Configurar matplotlib
plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Progreso del Entrenamiento SAC en CartPole-v1', fontsize=16, fontweight='bold')

# 1. Reward promedio por episodio
if 'env_runners/episode_return_mean' in trial_dataframe.columns:
    axes[0, 0].plot(trial_dataframe['training_iteration'], 
                   trial_dataframe['env_runners/episode_return_mean'], 
                   'b-', linewidth=2, label='Reward promedio')
    axes[0, 0].axhline(y=TRAINING_CONFIG["episode_reward_target"], 
                      color='r', linestyle='--', alpha=0.7, label=f'Objetivo ({TRAINING_CONFIG["episode_reward_target"]})')
    axes[0, 0].set_xlabel('Iteración de Entrenamiento')
    axes[0, 0].set_ylabel('Reward Promedio')
    axes[0, 0].set_title('Progreso del Reward')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

# 2. Longitud promedio de episodio
if 'env_runners/episode_len_mean' in trial_dataframe.columns:
    axes[0, 1].plot(trial_dataframe['training_iteration'], 
                   trial_dataframe['env_runners/episode_len_mean'], 
                   'g-', linewidth=2)
    axes[0, 1].set_xlabel('Iteración de Entrenamiento')
    axes[0, 1].set_ylabel('Longitud Promedio del Episodio')
    axes[0, 1].set_title('Duración de Episodios')
    axes[0, 1].grid(True, alpha=0.3)

# 3. Timesteps totales
if 'timesteps_total' in trial_dataframe.columns:
    axes[1, 0].plot(trial_dataframe['training_iteration'], 
                   trial_dataframe['timesteps_total'], 
                   'purple', linewidth=2)
    axes[1, 0].set_xlabel('Iteración de Entrenamiento')
    axes[1, 0].set_ylabel('Timesteps Totales')
    axes[1, 0].set_title('Progreso de Timesteps')
    axes[1, 0].grid(True, alpha=0.3)

# 4. Distribución de rewards de evaluación
axes[1, 1].bar(range(1, len(eval_rewards) + 1), eval_rewards, 
               color='skyblue', alpha=0.7, edgecolor='navy')
axes[1, 1].axhline(y=mean_reward, color='red', linestyle='-', 
                  linewidth=2, label=f'Promedio: {mean_reward:.1f}')
axes[1, 1].set_xlabel('Episodio de Evaluación')
axes[1, 1].set_ylabel('Reward Total')
axes[1, 1].set_title('Rewards de Evaluación')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("📊 Gráficos generados exitosamente!")

In [ ]:
# Resumen final y limpieza
print("=" * 60)
print("🎯 RESUMEN FINAL DEL ENTRENAMIENTO")
print("=" * 60)
print()
print(f"📋 Configuración:")
print(f"  • Algoritmo: SAC (Soft Actor-Critic)")
print(f"  • Entorno: {ENV_NAME}")
print(f"  • Framework: PyTorch")
print(f"  • Timesteps objetivo: {TRAINING_CONFIG['timesteps_total']:,}")
print()
print(f"📈 Resultados del entrenamiento:")
print(f"  • Timesteps completados: {final_metrics.get('timesteps_total', 'N/A'):,}")
print(f"  • Iteraciones: {final_metrics.get('training_iteration', 'N/A')}")
print(f"  • Duración: {training_duration:.1f} segundos")
print()
print(f"🎮 Rendimiento del agente:")
print(f"  • Reward promedio (entrenamiento): {achieved_reward:.2f}")
print(f"  • Reward promedio (evaluación): {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  • Problema resuelto: {'✅ Sí' if achieved_reward >= target_reward else '❌ No'}")
print()
print(f"💾 Datos guardados en:")
print(f"  • Experimento: {STORAGE_PATH}/{EXPERIMENT_NAME}")
print(f"  • Mejor checkpoint: {best_checkpoint_path}")
print()

# Opcional: Limpiar recursos
if input("¿Deseas cerrar Ray? (y/N): ").lower() == 'y':
    ray.shutdown()
    print("🔄 Ray cerrado correctamente")
else:
    print("🚀 Ray permanece activo para uso posterior")